In [3]:
from pathlib import Path
import os
import sys
from pyspark.sql import SparkSession

# Garantir que o pacote src seja encontrado quando o notebook for executado a partir da pasta do projeto.
if str(Path.cwd()) not in sys.path:
    sys.path.append(str(Path.cwd()))

try:
    from src.ingest.raw import metadata, write_postgres
except Exception:
    from ingest.raw import metadata, write_postgres

# ======================================================
# Spark Session
# ======================================================

spark = (
    SparkSession.builder
    .appName("Conexao PostgreSQL")
    .master("local[*]")
    .config("spark.sql.warehouse.dir", "/Users/eduardoalberto/LoadFile/output")
    .config("spark.jars", "/Users/eduardoalberto/opt/spark-3.5.4/jars/postgresql-42.7.5.jar")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

print(f"Spark Version : {spark.version}")

# ======================================================
# PostgreSQL
# ======================================================

POSTGRES = {
    "host": os.getenv("POSTGRES_HOST", "localhost"),
    "port": int(os.getenv("POSTGRES_PORT", "5432")),
    "database": os.getenv("POSTGRES_DATABASE", "dbpostgres"),
    "schema": "bronze",
    "user": os.getenv("POSTGRES_USER", "postgres"),
    "password": os.getenv("POSTGRES_PASSWORD", "postgre123"),
    "driver": "org.postgresql.Driver",
    "batchsize": 5000,
    "fetchsize": 1000,
    "numPartitions": 4,
}

print(f"Conectando ao PostgreSQL em {POSTGRES['host']}:{POSTGRES['port']} / {POSTGRES['database']} com usuário {POSTGRES['user']}")

STAGING = "/Users/eduardoalberto/LoadFile/staging"
PROCESSED = "/Users/eduardoalberto/LoadFile/processed"


Spark Version : 3.5.4
Conectando ao PostgreSQL em localhost:5432 / dbpostgres com usuário postgres


In [5]:
! ls -l /Users/eduardoalberto/LoadFile/staging/



total 0
drwxr-xr-x@ 38 eduardoalberto  staff  1216 Aug  3 20:57 csv
drwxr-xr-x@ 14 eduardoalberto  staff   448 Aug  3 21:05 kmz
drwxr-xr-x@  2 eduardoalberto  staff    64 Aug  3 20:57 ods


In [ ]:
# ======================================================
# Bulk Load CSV para PostgreSQL
# ======================================================

import re
from pathlib import Path
from pyspark.sql.functions import col, regexp_replace
from pyspark.sql.types import StringType

CSV_DIR = "/Users/eduardoalberto/LoadFile/staging/csv"


def sanitize_table_name(file_name):
    """Converte o nome do arquivo em um nome de tabela válido no PostgreSQL."""
    table_name = re.sub(r"[^a-zA-Z0-9_]+", "_", file_name).strip("_").lower()
    return table_name[:63]


def clean_null_bytes(dataframe):
    """Remove o byte nulo, que não é aceito pelo PostgreSQL em texto UTF-8."""
    string_columns = {
        field.name
        for field in dataframe.schema.fields
        if isinstance(field.dataType, StringType)
    }

    return dataframe.select(
        *[
            regexp_replace(col(column_name), "\\u0000", "").alias(column_name)
            if column_name in string_columns
            else col(column_name)
            for column_name in dataframe.columns
        ]
    )


def bulk_load_csv(csv_path, postgres_config):
    table_name = sanitize_table_name(csv_path.stem)
    qualified_table_name = f"{postgres_config['schema']}.{table_name}"
    print(f"Processando {csv_path.name} -> {qualified_table_name}")

    dataframe = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .option("encoding", "UTF-8")
        .option("escape", '"')
        .csv(str(csv_path))
    )
    dataframe = clean_null_bytes(dataframe)

    jdbc_url = (
        f"jdbc:postgresql://{postgres_config['host']}:{postgres_config['port']}"
        f"/{postgres_config['database']}"
    )

    (
        dataframe.write
        .format("jdbc")
        .mode("overwrite")
        .option("url", jdbc_url)
        .option("dbtable", qualified_table_name)
        .option("user", postgres_config["user"])
        .option("password", postgres_config["password"])
        .option("driver", postgres_config["driver"])
        .option("batchsize", postgres_config["batchsize"])
        .option("numPartitions", postgres_config["numPartitions"])
        .save()
    )

    return {"arquivo": csv_path.name, "tabela": qualified_table_name, "sucesso": True}


csv_files = sorted(Path(CSV_DIR).glob("*.csv"))
results = []

for csv_file in csv_files:
    try:
        results.append(bulk_load_csv(csv_file, POSTGRES))
    except Exception as error:
        table_name = sanitize_table_name(csv_file.stem)
        qualified_table_name = f"{POSTGRES['schema']}.{table_name}"
        print(f"Erro ao processar {csv_file.name}: {error}")
        results.append({"arquivo": csv_file.name, "tabela": qualified_table_name, "sucesso": False})

print("\nResumo do bulk load:")
for result in results:
    status = "Sucesso" if result["sucesso"] else "Erro"
    print(f"{status}: {result['arquivo']} -> {result['tabela']}")

sucessos = sum(result["sucesso"] for result in results)
print(f"Total: {sucessos}/{len(results)} arquivo(s) carregado(s) com sucesso")

Processando bancocrai2014a2024-sistematizacao-geoinfo-atualizada.csv -> bronze.bancocrai2014a2024_sistematizacao_geoinfo_atualizada
Processando base-de-dados-crai-a-partir-de-2025.csv -> bronze.base_de_dados_crai_a_partir_de_2025
Processando basecadunicoimigrantesmsp082020.csv -> bronze.basecadunicoimigrantesmsp082020
Processando dicionario-base-de-dados-crai-a-partir-de-2025.csv -> bronze.dicionario_base_de_dados_crai_a_partir_de_2025
Processando dicionario-de-bancocrai2014a2019.csv -> bronze.dicionario_de_bancocrai2014a2019
Processando dicionariobasecadunicoimigrantesmsp082020-1.csv -> bronze.dicionariobasecadunicoimigrantesmsp082020_1
Processando pab_2022_01.csv -> bronze.pab_2022_01
Processando pab_2022_1.csv -> bronze.pab_2022_1
Processando pab_jan_2020.csv -> bronze.pab_jan_2020
Processando pab_jan_2021.csv -> bronze.pab_jan_2021
Processando pbf_2007.csv -> bronze.pbf_2007
Processando pbf_2007_kml.csv -> bronze.pbf_2007_kml
Processando pbf_2010.csv -> bronze.pbf_2010
Processando 

26/08/22 14:43:01 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$driverEndpoint(BlockManagerMasterEndpoint.scala:123)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.isExecutorAlive$lzycompute$1(BlockManagerMasterEndpoint.scala:688)
	at org.apache.spark.storage.BlockManagerMasterE